In [1]:
# BertViz + Transformers 설치
!pip -q install "transformers>=4.41" "bertviz>=1.4.1" sentencepiece

from google.colab import output
output.enable_custom_widget_manager()
print("✅ Ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.8 MB/s eta 0:00:00
✅ Ready.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModel
from bertviz import head_view, model_view

# 1) BERT 불러오기 (어텐션 반환 옵션)
tok_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

# 2) 짧은 문장
sent = "The cat sat on the mat."
ids = tok_bert.encode(sent, return_tensors="pt")
with torch.no_grad():
    out = bert(ids)  # out[-1] 또는 out.attentions에 레이어별 어텐션

attn = out[-1]
tokens = tok_bert.convert_ids_to_tokens(ids[0])

print(tokens)              # 확인용
head_view(attn, tokens)    # 단층/복수 헤드 연결(멀티헤드는 상단 체크로 On/Off)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'the', 'cat', 'sat', 'on', 'the', 'mat', '.', '[SEP]']


<IPython.core.display.Javascript object>

In [3]:
import torch
from transformers import AutoTokenizer, AutoModel
from bertviz import head_view

tok_gpt2 = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModel.from_pretrained("gpt2", output_attentions=True)

sent_gen = "She sees the small elephant."
ids = tok_gpt2.encode(sent_gen, return_tensors="pt")

with torch.no_grad():
    out = gpt2(ids)
attn = out[-1]
tokens = tok_gpt2.convert_ids_to_tokens(ids[0])

head_view(attn, tokens)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<IPython.core.display.Javascript object>

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from bertviz import model_view

# 한→영 번역용 MarianMT 모델
name = "Helsinki-NLP/opus-mt-ko-en"
tok_mt = AutoTokenizer.from_pretrained(name)
mt = AutoModelForSeq2SeqLM.from_pretrained(name, output_attentions=True)

# 원문(한글) & 번역문(영어)
src = "나는 학교에 간다."
tgt = "I go to school."

# 인코더 입력
enc_ids = tok_mt(src, return_tensors="pt", add_special_tokens=True).input_ids

# 디코더 입력 (정답 문장을 teacher forcing으로 넣음)
dec_ids = tok_mt(text_target=tgt, return_tensors="pt", add_special_tokens=True).input_ids

# 추론 실행 → 어텐션 포함
with torch.no_grad():
    out = mt(input_ids=enc_ids, decoder_input_ids=dec_ids)

# 토큰 복원
enc_toks = tok_mt.convert_ids_to_tokens(enc_ids[0])
dec_toks = tok_mt.convert_ids_to_tokens(dec_ids[0])

# BertViz 시각화: 드롭다운에서 Encoder / Decoder / Cross 선택
model_view(
    encoder_attention=out.encoder_attentions,
    decoder_attention=out.decoder_attentions,
    cross_attention=out.cross_attentions,
    encoder_tokens=enc_toks,
    decoder_tokens=dec_toks
)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/842k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

<IPython.core.display.Javascript object>

In [5]:
# 👇 BertViz의 Neuron View는 'Q/K 뉴런'에 접근해야 하므로
#    Hugging Face AutoModel이 아니라 'bertviz 전용 래퍼'를 사용합니다.
from bertviz.transformers_neuron_view import BertModel, BertTokenizer  # 전용 래퍼
from bertviz.neuron_view import show

# 1) 모델/토크나이저 로드
model_type   = "bert"
model_name = "bert-base-uncased"
model = BertModel.from_pretrained(model_name, output_attentions=True)
tokenizer = BertTokenizer.from_pretrained(model_name, do_lower_case=True)

# 2) 단일 문장
sentence = "The cat sat on the mat."

# 3) Neuron View 시각화
show(
    model,                 # BertViz 전용 BERT 모델
    model_type,                # model_type: 'bert' | 'gpt2' | 'roberta'
    tokenizer,             # 전용 토크나이저
    sentence_a=sentence,   # 단일 문장만 사용 (비교용 sentence_b는 생략)
    layer=2,               # 초기 레이어 (예: 3번째 레이어)
    head=0,                # 초기 헤드 (예: 1번째 헤드)
)

100%|██████████| 231508/231508 [00:00<00:00, 4230485.74B/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>